# Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
get_ipython().system('ls /content/drive/MyDrive')

# ShanghaiTech Dataset Part A and Part B

In [ ]:
# STEP 1A — Dataset locator & normalizer (Colab)
# - Mounts Drive
# - Searches for ShanghaiTech in /content and /content/drive/MyDrive
# - Accepts these shapes: part_A_final / part_B_final OR ShanghaiTechA / ShanghaiTechB
# - If a ZIP is found, unzips & normalizes
# - Prints final counts

import os, glob, shutil, zipfile, re, scipy.io as sio

# 0) Mount Drive
try:
    from google.colab import drive as _gdrive
    _gdrive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("[warn] Could not mount Drive (ok if you only use /content):", e)

ROOTS = [
    "/content",
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/datasets",
    "/content/drive/MyDrive/Downloads",
]

TARGET_ROOT = "/content/ShanghaiTech"
os.makedirs(TARGET_ROOT, exist_ok=True)

def _norm_copy(src_dir, dst_dir):
    if os.path.abspath(src_dir) == os.path.abspath(dst_dir):
        return
    os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
    # move if inside /content; copy if from Drive
    try:
        if src_dir.startswith("/content/drive"):
            shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        else:
            shutil.move(src_dir, dst_dir)
    except Exception as e:
        print(f"[warn] move/copy failed {src_dir} -> {dst_dir}:", e)

def _unzip_here(zip_path, out_dir):
    print(f"[info] Unzipping {zip_path} ...")
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)

def _part_stats(root):
    imgs_tr = glob.glob(os.path.join(root, "train_data/images/*.jpg"))
    gts_tr  = glob.glob(os.path.join(root, "train_data/ground_truth/*.mat"))
    imgs_te = glob.glob(os.path.join(root, "test_data/images/*.jpg"))
    gts_te  = glob.glob(os.path.join(root, "test_data/ground_truth/*.mat"))
    return len(imgs_tr), len(gts_tr), len(imgs_te), len(gts_te)

# 1) Try to locate folders or zips
cand_dirs = []
cand_zips = []
for r in ROOTS:
    cand_dirs += glob.glob(r + "/**/ShanghaiTechA", recursive=True)
    cand_dirs += glob.glob(r + "/**/ShanghaiTechB", recursive=True)
    cand_dirs += glob.glob(r + "/**/part_A_final", recursive=True)
    cand_dirs += glob.glob(r + "/**/part_B_final", recursive=True)
    cand_zips += glob.glob(r + "/**/*ShanghaiTech*Crowd*Counting*Dataset*.zip", recursive=True)
    cand_zips += glob.glob(r + "/**/*ShanghaiTech*Dataset*.zip", recursive=True)

print(f"[scan] found dirs: {len(cand_dirs)}, zips: {len(cand_zips)}")

# 2) If zip exists and target missing, unzip first
A_TGT = os.path.join(TARGET_ROOT, "ShanghaiTechA")
B_TGT = os.path.join(TARGET_ROOT, "ShanghaiTechB")

if (not os.path.exists(A_TGT) or not os.path.exists(B_TGT)) and cand_zips:
    _unzip_here(cand_zips[0], TARGET_ROOT)

# 3) Normalize possible names to target
# priorities: explicit ShanghaiTechA/B, else part_A_final/B
def _find_and_place(name_patterns, dst):
    for d in cand_dirs + glob.glob(TARGET_ROOT + "/**/*", recursive=True):
        for pat in name_patterns:
            if d.endswith(pat) and os.path.isdir(d):
                print(f"[place] {d} -> {dst}")
                _norm_copy(d, dst)
                return True
    return False

if not os.path.exists(A_TGT):
    okA = _find_and_place(["ShanghaiTechA", "part_A_final"], A_TGT)
else:
    okA = True

if not os.path.exists(B_TGT):
    okB = _find_and_place(["ShanghaiTechB", "part_B_final"], B_TGT)
else:
    okB = True

# 4) Final stats
def peek_count_example(root):
    imgs = sorted(glob.glob(os.path.join(root, "train_data/images/*.jpg")))
    if not imgs: return None
    ex = imgs[0]
    gt = ex.replace("images","ground_truth").replace("IMG_","GT_IMG_").replace(".jpg",".mat")
    pts = sio.loadmat(gt)["image_info"][0,0][0,0][0]
    return os.path.basename(ex), int(pts.shape[0])

stats = {}
if okA and os.path.isdir(A_TGT):
    a_tr,a_gt,a_te,a_gt2 = _part_stats(A_TGT); stats["A"]=(a_tr,a_gt,a_te,a_gt2)
if okB and os.path.isdir(B_TGT):
    b_tr,b_gt,b_te,b_gt2 = _part_stats(B_TGT); stats["B"]=(b_tr,b_gt,b_te,b_gt2)

print("\n=== FINAL LAYOUT ===")
print("A ->", A_TGT, stats.get("A","(missing)"))
print("B ->", B_TGT, stats.get("B","(missing)"))

exA = peek_count_example(A_TGT) if os.path.isdir(A_TGT) else None
exB = peek_count_example(B_TGT) if os.path.isdir(B_TGT) else None
if exA: print("[A] sample:", exA)
if exB: print("[B] sample:", exB)

assert os.path.isdir(A_TGT) and os.path.isdir(B_TGT), "Dataset still not found. Upload/unzip or point PATHs correctly."
print("✅ ShanghaiTech is ready at /content/ShanghaiTech/{ShanghaiTechA,ShanghaiTechB}")


# Implementation Of YoloV12 Baseline

In [ ]:
%pip install -q ultralytics


In [ ]:
# STEP 2 — YOLOv12 baseline detect+count (single image)

from ultralytics import YOLO
import cv2, numpy as np, time
from google.colab import files
from matplotlib import pyplot as plt

# classes we care about
INTEREST_CLASSES = ["person","bicycle","car","motorbike","bus","truck","dog","cat"]

# load a reasonably strong YOLOv12 model (swap to 'yolo12m.pt' if GPU is strong)
YOLO_WEIGHTS = "yolo12s.pt"
yolo = YOLO(YOLO_WEIGHTS)
_ = yolo.predict(np.zeros((640,640,3),dtype=np.uint8), imgsz=640, conf=0.25, verbose=False)  # warm-up

# inference params (good recall for small people)
IMG_SIZE   = 1280
CONF_THRES = 0.15
IOU_THRES  = 0.60
MAX_DET    = 3000

def yolo_detect_and_count(img_bgr):
    res = yolo.predict(
        img_bgr, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES,
        max_det=MAX_DET, verbose=False
    )[0]
    # tally + draw
    counts = {k:0 for k in INTEREST_CLASSES}
    vis = img_bgr.copy()
    for b, c, conf in zip(res.boxes.xyxy.cpu().numpy(),
                          res.boxes.cls.cpu().numpy(),
                          res.boxes.conf.cpu().numpy()):
        cname = yolo.names[int(c)]
        if cname in counts:
            counts[cname] += 1
            x1,y1,x2,y2 = b.astype(int)
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(vis,f"{cname} {conf:.2f}",(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
    # HUD
    y=28
    cv2.putText(vis,f"YOLOv12 persons: {counts['person']}",(12,y),cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2); y+=28
    for k in INTEREST_CLASSES:
        if k=="person": continue
        if counts[k]>0:
            cv2.putText(vis,f"{k}: {counts[k]}",(12,y),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0),2); y+=24
    return vis, counts

def analyze_one_image(path=None):
    # choose an image: if None → open uploader; else use given path
    if path is None:
        up = files.upload()
        if not up:
            print("No file uploaded.");
            return
        path = list(up.keys())[0]
    img = cv2.imread(path); assert img is not None, f"failed to read {path}"
    vis, counts = yolo_detect_and_count(img)
    print("Counts:", {k:v for k,v in counts.items() if v>0})
    plt.figure(figsize=(10,6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
    return counts

# --- try it ---
# Option A: run on a Part-B test image (change path as you like)
# analyze_one_image("/content/ShanghaiTech/ShanghaiTechB/test_data/images/IMG_144.jpg")

# Option B: upload any image
# analyze_one_image()


# Implementation of CSRNet framework for high Dense Crowd

In [ ]:
# STEP 3 — CSRNet-Lite + GAK dataset (sanity check only)

import os, glob, cv2, math, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import NearestNeighbors
from matplotlib import pyplot as plt

# ---- paths (from Step 1A) ----
PATH_A = "/content/ShanghaiTech/ShanghaiTechA"
PATH_B = "/content/ShanghaiTech/ShanghaiTechB"

# ---- helpers ----
def load_points(mat_path):
    m = sio.loadmat(mat_path)
    return m["image_info"][0,0][0,0][0]  # (N,2)

def make_density_gak_norm(h, w, points, k=3, beta=0.3):
    """Geometry-Adaptive Kernel with per-crop renormalization (sum≈#heads)."""
    den = np.zeros((h, w), dtype=np.float32)
    if len(points) == 0:
        return den
    pts = np.asarray(points, dtype=np.float32)
    xs = np.clip(pts[:,0], 0, w-1).astype(int)
    ys = np.clip(pts[:,1], 0, h-1).astype(int)
    if len(pts) > 1:
        nbrs = NearestNeighbors(n_neighbors=min(k+1, len(pts))).fit(pts)
        dists, _ = nbrs.kneighbors(pts)
        sigmas = beta * dists[:,1:].mean(axis=1)  # skip self
    else:
        sigmas = np.full((len(pts),), 15.0, dtype=np.float32)
    for (x, y), s in zip(np.stack([xs,ys], axis=1), sigmas):
        s = max(1.0, float(s))
        r = int(3*s)
        x0, x1 = max(0, x-r), min(w, x+r+1)
        y0, y1 = max(0, y-r), min(h, y+r+1)
        if x0>=x1 or y0>=y1:
            continue
        xs2 = np.arange(x0, x1) - x
        ys2 = np.arange(y0, y1) - y
        gx = np.exp(-(xs2**2)/(2*s*s))
        gy = np.exp(-(ys2**2)/(2*s*s))
        g = np.outer(gy, gx).astype(np.float32)
        ssum = g.sum()
        if ssum > 0: g /= ssum  # renorm so each truncated kernel sums to ~1
        den[y0:y1, x0:x1] += g
    return den

class STDataset(Dataset):
    def __init__(self, root, split="train", part="B", img_size=512, aug=False):
        self.img_paths = sorted(glob.glob(os.path.join(root, f"{split}_data/images/*.jpg")))
        self.gt_paths  = [p.replace("images","ground_truth").replace("IMG_","GT_IMG_").replace(".jpg",".mat")
                          for p in self.img_paths]
        self.part = part
        self.img_size = img_size
        self.aug = aug and (split=="train")

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, i):
        img_path = self.img_paths[i]
        gt_path  = self.gt_paths[i]
        img_bgr = cv2.imread(img_path); assert img_bgr is not None, img_path
        h0, w0 = img_bgr.shape[:2]

        # keep aspect resize so max side = img_size
        scale = self.img_size / max(h0, w0)
        nh, nw = int(h0*scale), int(w0*scale)
        img_bgr = cv2.resize(img_bgr, (nw, nh))
        pts = load_points(gt_path).copy()
        if len(pts)>0:
            pts[:,0] *= nw / w0
            pts[:,1] *= nh / h0

        # (no heavy aug for the sanity step)

        den = make_density_gak_norm(nh, nw, pts, k=3, beta=0.3)

        # to tensors (contiguous, RGB)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_t = torch.from_numpy(np.ascontiguousarray(img_rgb).transpose(2,0,1)).float()/255.0
        den_t = torch.from_numpy(den).unsqueeze(0)  # [1,h,w]
        cnt_t = torch.tensor([float(den_t.sum())], dtype=torch.float32)  # GT headcount

        return img_t, den_t, cnt_t, os.path.basename(img_path)

# ---- model (CSRNet-Lite) ----
class ConvReLU(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=1, d=1):
        super().__init__()
        self.c = nn.Conv2d(c1,c2,k,s,p,dilation=d)
        self.b = nn.BatchNorm2d(c2)
    def forward(self,x): return F.relu(self.b(self.c(x)), inplace=True)

class CSRNetLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            ConvReLU(3, 16), ConvReLU(16,16), nn.MaxPool2d(2),
            ConvReLU(16, 32), ConvReLU(32,32), nn.MaxPool2d(2),
            ConvReLU(32, 64), ConvReLU(64,64), ConvReLU(64,64)
        )
        self.b = nn.Sequential(
            ConvReLU(64,128,d=2,p=2),
            ConvReLU(128,128,d=2,p=2),
            ConvReLU(128,64,d=2,p=2),
            nn.Conv2d(64,1,1)
        )
    def forward(self,x):
        x = self.f(x)
        x = self.b(x)
        return F.relu(x, inplace=True)  # density map

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- build a tiny loader (Part B) for sanity check ----
train_ds = STDataset(PATH_B, split="train", part="B", img_size=512, aug=False)
test_ds  = STDataset(PATH_B, split="test",  part="B", img_size=512, aug=False)
print(f"[Part B] train={len(train_ds)}, test={len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)

# ---- model & one forward pass ----
model = CSRNetLite().to(device)
img, den, cnt, name = next(iter(train_loader))
img, den = img.to(device), den.to(device)
with torch.no_grad():
    pred = model(img)             # [B,1,Hp,Wp]

# sum-preserving note: during training we will resize GT to pred size and
# multiply by area_ratio so sums match. Here we only print sums for sanity.
gt_sum = float(den.sum().item())
pred_sum = float(pred.sum().item())

print(f"[sanity] batch={img.shape[0]} | GT sum (heads) = {gt_sum:.1f} | Pred sum (raw) = {pred_sum:.1f}")
print("[sanity] pred map size:", list(pred.shape), "| gt map size:", list(den.shape))

# quick viz for one sample
i = 0
dm = pred[i,0].detach().cpu().numpy()
dm = (dm - dm.min()) / (np.ptp(dm) + 1e-6)
dm_color = cv2.applyColorMap((dm*255).astype(np.uint8), cv2.COLORMAP_JET)
im = img[i].permute(1,2,0).cpu().numpy()
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.title(name[i]); plt.imshow(im); plt.axis('off')
plt.subplot(1,3,2); plt.title("pred density (raw)"); plt.imshow(dm_color[:,:,::-1]); plt.axis('off')
plt.subplot(1,3,3); plt.title(f"GT count≈{cnt[i].item():.1f}"); plt.imshow(den[i,0].cpu().numpy(), cmap='jet'); plt.axis('off')
plt.show()


# Training CSRNet-Lite with sum-preserving loss + MAE/RMSE eval (Part B)

In [ ]:
# STEP 4 — Train CSRNet-Lite with sum-preserving loss + MAE/RMSE eval (Part B)

import math, time, torch
from torch.utils.data import DataLoader

# you already have: train_ds, test_ds, model, device
BATCH   = 4          # drop to 2 if OOM
EPOCHS  = 20        # start with 20; increase to 50–80 for strong accuracy
LR      = 1e-4
WD      = 1e-5
CLIP    = 5.0        # grad clip (helps stability)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=1,    shuffle=False, num_workers=0, pin_memory=False)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
mse = torch.nn.MSELoss(reduction="mean")
best_mae = float("inf")
best_path = "/content/weights_csrnet_lite_stb_best.pt"

def train_one_epoch(epoch):
    model.train()
    running = 0.0
    n_samp = 0
    t0 = time.time()
    for img, den, _, _ in train_loader:
        img, den = img.to(device), den.to(device)          # den: [B,1,Hgt,Wgt]
        pred = model(img)                                  # pred: [B,1,Hp,Wp]
        # --- sum-preserving resize of GT ---
        Hgt, Wgt = den.shape[-2], den.shape[-1]
        Hp,  Wp  = pred.shape[-2], pred.shape[-1]
        den_rs = torch.nn.functional.interpolate(den, size=(Hp, Wp), mode="bilinear", align_corners=False)
        scale = (Hgt * Wgt) / (Hp * Wp)                    # preserve ∑density
        den_rs = den_rs * scale

        loss = mse(pred, den_rs)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        opt.step()

        bs = img.size(0)
        running += loss.item() * bs
        n_samp += bs

    dt = time.time() - t0
    return running / max(1,n_samp), dt

@torch.no_grad()
def eval_mae_rmse():
    model.eval()
    abs_err, sq_err, n = 0.0, 0.0, 0
    for img, den, _, _ in test_loader:
        img, den = img.to(device), den.to(device)          # den is ORIGINAL GT
        pred = model(img)
        c_pred = float(pred.sum().item())
        c_gt   = float(den.sum().item())
        abs_err += abs(c_pred - c_gt)
        sq_err  += (c_pred - c_gt) ** 2
        n += 1
    mae  = abs_err / n
    rmse = math.sqrt(sq_err / n)
    return mae, rmse

print(f"Training on device: {device} | epochs={EPOCHS} | batch={BATCH}")
for ep in range(1, EPOCHS+1):
    tl, dt = train_one_epoch(ep)
    mae, rmse = eval_mae_rmse()
    is_best = mae < best_mae
    if is_best:
        best_mae = mae
        torch.save(model.state_dict(), best_path)
    print(f"ep {ep:3d}/{EPOCHS}  loss {tl:.4f}  |  val MAE {mae:.2f}  RMSE {rmse:.2f}  "
          f"| best MAE {best_mae:.2f}  | {dt:.1f}s")

print(f"✅ Done. Best weights saved → {best_path}")


# Best weights saved → /content/weights_csrnet_lite_stb_best.pt

In [ ]:
!cp weights_csrnet_lite_stb_best.pt /content/drive/MyDrive/weights_csrnet_lite_stb_best.pt


In [ ]:
CSRNET_WEIGHTS = "/content/weights_csrnet_lite_stb_best.pt"

density_net = CSRNetLite().to(device)
density_net.load_state_dict(torch.load(CSRNET_WEIGHTS, map_location=device))
density_net.eval()

print("Loaded CSRNet-Lite weights successfully!")


In [ ]:
dm, cnt = density_count_bgr(cv2.imread("/content/ShanghaiTech/ShanghaiTechB/test_data/images/IMG_144.jpg"))
print("Density count =", cnt)


# Addition of Alerts and MQTT protocols

In [ ]:
# STEP 5 — Inference + Fusion + Alerts (single-image ready)

import os, cv2, time, json, numpy as np, requests
import torch, torch.nn as nn, torch.nn.functional as F
from ultralytics import YOLO
from matplotlib import pyplot as plt

# ========= Config =========
CSRNET_WEIGHTS = "/content/weights_csrnet_lite_stb_best.pt"  # from Step 4 output
DENSITY_IMG_SIZE = 512

# YOLO settings (good recall)
YOLO_WEIGHTS = "yolo12s.pt"   # use 'yolo12m.pt' if you have GPU
IMG_SIZE   = 1280
CONF_THRES = 0.15
IOU_THRES  = 0.60
MAX_DET    = 3000

# Classes of interest
INTEREST_CLASSES = ["person","bicycle","car","motorbike","bus","truck","dog","cat"]
ANIMAL_CLASSES   = {"dog","cat","bird","horse","sheep","cow","elephant","bear","zebra","giraffe"}

# Fusion thresholds
SWITCH_TO_DENSITY = 10.0
DENSITY_DOMINATES_FACTOR = 1.2
DENSITY_MIN_VALID = 5.0

# Alerting
PERSON_THRESHOLD_GLOBAL = 50
ALERT_ON_ANIMAL_IN_CROWD = True
ALERT_COOLDOWN_SEC = 60
BOT_TOKEN = ""  # optional: paste Telegram bot token
CHAT_ID   = 0   # optional: paste chat id (int)
USE_MQTT  = False
MQTT_HOST, MQTT_PORT, MQTT_TOPIC_BASE = "broker.emqx.io", 1883, "site/demo/camera/colab"

# ========= Models =========
device = "cuda" if torch.cuda.is_available() else "cpu"

# CSRNet-Lite (same architecture as Step 3/4)
class ConvReLU(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=1, d=1):
        super().__init__()
        self.c = nn.Conv2d(c1,c2,k,s,p,dilation=d)
        self.b = nn.BatchNorm2d(c2)
    def forward(self,x): return F.relu(self.b(self.c(x)), inplace=True)

class CSRNetLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            ConvReLU(3, 16), ConvReLU(16,16), nn.MaxPool2d(2),
            ConvReLU(16, 32), ConvReLU(32,32), nn.MaxPool2d(2),
            ConvReLU(32, 64), ConvReLU(64,64), ConvReLU(64,64)
        )
        self.b = nn.Sequential(
            ConvReLU(64,128,d=2,p=2),
            ConvReLU(128,128,d=2,p=2),
            ConvReLU(128,64,d=2,p=2),
            nn.Conv2d(64,1,1)
        )
    def forward(self,x):
        x = self.f(x)
        x = self.b(x)
        return F.relu(x, inplace=True)

density_net = CSRNetLite().to(device)
density_net.load_state_dict(torch.load(CSRNET_WEIGHTS, map_location=device))
density_net.eval()

# YOLOv12
yolo = YOLO(YOLO_WEIGHTS)
_ = yolo.predict(np.zeros((640,640,3),dtype=np.uint8), imgsz=640, conf=0.25, verbose=False)  # warm-up

# ========= Networking helpers =========
_last_alert = {}
def should_alert(key):
    t = time.time(); last = _last_alert.get(key, 0.0)
    if t - last >= ALERT_COOLDOWN_SEC:
        _last_alert[key] = t; return True
    return False

def send_telegram(text, photo_bgr=None):
    if not BOT_TOKEN or not CHAT_ID:
        print("[Alert]", text); return
    try:
        if photo_bgr is None:
            requests.post(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
                          json={"chat_id": CHAT_ID, "text": text}, timeout=5)
        else:
            ok, buf = cv2.imencode(".jpg", photo_bgr)
            if ok:
                files={"photo": ("alert.jpg", buf.tobytes(), "image/jpeg")}
                data={"chat_id": CHAT_ID, "caption": text}
                requests.post(f"https://api.telegram.org/bot{BOT_TOKEN}/sendPhoto",
                              data=data, files=files, timeout=10)
    except Exception as e:
        print("Telegram error:", e)

if USE_MQTT:
    from paho.mqtt import client as mqtt
    mq = mqtt.Client(client_id="crowd_colab")
    try:
        mq.connect(MQTT_HOST, MQTT_PORT); mq.loop_start()
    except Exception as e:
        print("MQTT connect error:", e); USE_MQTT=False

def publish_mqtt(topic_suffix, payload:dict):
    if not USE_MQTT: return
    try:
        mq.publish(f"{MQTT_TOPIC_BASE}/{topic_suffix}", json.dumps(payload), qos=0)
    except Exception as e:
        print("MQTT publish error:", e)

# ========= Inference helpers =========
def density_count_bgr(frame_bgr, input_size=DENSITY_IMG_SIZE):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h0,w0 = rgb.shape[:2]
    scale = input_size / max(h0,w0)
    nh,nw = int(h0*scale), int(w0*scale)
    rgb = cv2.resize(rgb, (nw, nh))
    ten = torch.from_numpy(np.ascontiguousarray(rgb).transpose(2,0,1)).float().unsqueeze(0)/255.0
    ten = ten.to(device)
    with torch.no_grad():
        dm = density_net(ten)             # [1,1,H,W]
    count = float(dm.sum().item())
    return dm.squeeze().cpu().numpy(), count

def run_yolo(frame_bgr):
    res = yolo.predict(frame_bgr, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES,
                       max_det=MAX_DET, verbose=False)[0]
    dets=[]; counts={k:0 for k in INTEREST_CLASSES}
    for b,c,conf in zip(res.boxes.xyxy.cpu().numpy(),
                        res.boxes.cls.cpu().numpy(),
                        res.boxes.conf.cpu().numpy()):
        cname = yolo.names[int(c)]
        x1,y1,x2,y2 = b.astype(int)
        cx,cy = int((x1+x2)//2), int((y1+y2)//2)
        dets.append({"cls":cname,"conf":float(conf),"bbox":[x1,y1,x2,y2],"center":[cx,cy]})
        if cname in counts: counts[cname]+=1
    return dets, counts

def fused_people_count(yolo_persons:int, frame_bgr):
    _, d = density_count_bgr(frame_bgr)
    y = float(yolo_persons); d = float(d)
    use_density = (d >= max(DENSITY_MIN_VALID, SWITCH_TO_DENSITY)) or (d > y * DENSITY_DOMINATES_FACTOR)
    return (d, "density") if use_density else (y, "yolo")

# ========= Single-image analysis =========
from google.colab import files

def analyze_image(path=None, trigger_alerts=True, show=True):
    # choose image
    if path is None:
        up = files.upload()
        if not up: print("No file uploaded."); return
        path = list(up.keys())[0]
    img = cv2.imread(path); assert img is not None, f"failed to read {path}"

    # YOLO
    dets, counts_det = run_yolo(img)

    # Fused persons
    fused_count, source_name = fused_people_count(counts_det.get("person",0), img)

    # draw
    vis = img.copy()
    for d in dets:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)
    cv2.putText(vis,f"Persons (fused-{source_name}): {fused_count:.1f}",(12,28),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2)

    # alerts
    if trigger_alerts:
        # threshold
        if fused_count >= PERSON_THRESHOLD_GLOBAL and should_alert("IMG_person"):
            msg=f"🚨 Crowd Alert: persons={fused_count:.1f} (≥{PERSON_THRESHOLD_GLOBAL})"
            send_telegram(msg, photo_bgr=vis)
            publish_mqtt("alerts", {"type":"img_person_threshold","value":fused_count,"thr":PERSON_THRESHOLD_GLOBAL,"ts":time.time()})
        # animal-in-crowd
        if ALERT_ON_ANIMAL_IN_CROWD and fused_count>=1:
            animals = [k for k,v in counts_det.items() if k in ANIMAL_CLASSES and v>0]
            if animals and should_alert("IMG_animal"):
                msg=f"⚠️ Animal in crowd: {', '.join(animals)} | persons≈{fused_count:.1f}"
                send_telegram(msg, photo_bgr=vis)
                publish_mqtt("alerts", {"type":"img_animal_in_crowd","animals":animals,"people":fused_count,"ts":time.time()})

    # show
    if show:
        plt.figure(figsize=(10,6))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

    print("Counts:", {k:v for k,v in counts_det.items() if v>0})
    print(f"Fused persons: {fused_count:.1f} (source: {source_name})")
    return {"counts_det": counts_det, "fused": fused_count, "source": source_name}

# --- usage examples ---
# analyze_image("/content/ShanghaiTech/ShanghaiTechB/test_data/images/IMG_144.jpg")
# analyze_image()  # upload an image


# Quick test on Part-B dataset  

In [ ]:
# QUICK TEST — pick a Part-B test image and run analysis
import glob, os

# try Part-B test set first; fallback to Part-A; else prompt upload
candidates = glob.glob("/content/ShanghaiTech/ShanghaiTechB/test_data/images/*.jpg")
if not candidates:
    candidates = glob.glob("/content/ShanghaiTech/ShanghaiTechA/test_data/images/*.jpg")

if candidates:
    TEST_IMG = candidates[0]
    print("Testing on:", TEST_IMG)
    out = analyze_image(TEST_IMG)   # calls the function from Step 5 and shows the figure
else:
    print("No dataset images found — opening uploader…")
    out = analyze_image()           # will prompt you to upload one

print("Result:", out)


# Video Slicing (YOLOv12 + density fusion + alerts)

In [ ]:
# STEP 6 — Video/RTSP loop (YOLOv12 + density fusion + alerts)

import cv2, time, numpy as np, glob
from google.colab import files
from google.colab.patches import cv2_imshow

# Reuse: yolo, density_count_bgr, run_yolo, fused_people_count, send_telegram, publish_mqtt, should_alert
# (already defined in Step 5)

def pick_source():
    # try to find any mp4 in /content
    vids = glob.glob("/content/*.mp4") + glob.glob("/content/**/*.mp4", recursive=True)
    if vids:
        print("[source] Using found video:", vids[0])
        return vids[0]
    print("[source] No video found. Please upload one (mp4) or paste an RTSP/URL string below.")
    up = files.upload()
    if up:
        name = list(up.keys())[0]
        print("[source] Uploaded:", name)
        return name
    raise RuntimeError("No video provided.")

def process_stream(source=None, show_every=3, max_frames=None):
    """
    source: path/URL (mp4/rtsp). If None, prompts upload or auto-picks a video.
    show_every: display every Nth frame to keep UI responsive
    max_frames: cap for quick tests; None to run full video
    """
    if source is None:
        source = pick_source()

    # RTSP tip: OpenCV sometimes needs FFMPEG flags; basic VideoCapture should work for mp4/rtsp in Colab
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise AssertionError(f"Could not open source: {source}")

    t0 = time.time()
    n = 0
    fused_hist = []
    alert_hits = 0

    print(f"[run] reading from: {source}")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        n += 1
        vis = frame.copy()

        # YOLO detections
        dets, counts_det = run_yolo(frame)

        # draw detections
        for d in dets:
            x1,y1,x2,y2 = d["bbox"]
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)

        # fused person count
        fused_count, source_name = fused_people_count(counts_det.get("person",0), frame)
        fused_hist.append(fused_count)

        # HUD
        y=28
        cv2.putText(vis,f"Persons (fused-{source_name}): {fused_count:.1f}",(12,y),
                    cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2); y+=28
        for k in INTEREST_CLASSES:
            if k=="person": continue
            c = counts_det.get(k,0)
            if c>0:
                cv2.putText(vis,f"{k}: {c}",(12,y),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,0),2); y+=24

        # alerts
        if fused_count >= PERSON_THRESHOLD_GLOBAL and should_alert("VID_person"):
            alert_hits += 1
            msg=f"🚨 Crowd Alert: persons={fused_count:.1f} (≥{PERSON_THRESHOLD_GLOBAL})"
            send_telegram(msg, photo_bgr=vis)
            publish_mqtt("alerts", {"type":"video_person_threshold","value":float(fused_count),"thr":PERSON_THRESHOLD_GLOBAL,"ts":time.time()})

        if ALERT_ON_ANIMAL_IN_CROWD and fused_count>=1:
            animals = [k for k,v in counts_det.items() if k in ANIMAL_CLASSES and v>0]
            if animals and should_alert("VID_animal"):
                alert_hits += 1
                msg=f"⚠️ Animal in crowd: {', '.join(animals)} | persons≈{fused_count:.1f}"
                send_telegram(msg, photo_bgr=vis)
                publish_mqtt("alerts", {"type":"video_animal_in_crowd","animals":animals,"people":float(fused_count),"ts":time.time()})

        # show sampled frames
        if (n % show_every) == 0:
            cv2_imshow(vis)

        if max_frames and n >= max_frames:
            break

    cap.release()
    dt = time.time()-t0
    avg = float(np.mean(fused_hist)) if fused_hist else 0.0
    print(f"[done] frames={n} | avg fused persons={avg:.1f} | alerts sent={alert_hits} | {dt:.1f}s total")

# --- Usage:
# process_stream("/content/your_video.mp4", show_every=3)   # specific file
# process_stream(None, show_every=3)                        # auto-pick or upload
# process_stream("rtsp://user:pass@IP:554/stream", show_every=5)


In [ ]:
process_stream(None, show_every=3)  # it will ask for/upload or auto-pick a video


# Per-zone polygons + per-zone thresholds + zone alerts

In [ ]:
# STEP 7 — Per-zone polygons + per-zone thresholds + zone alerts

import cv2, numpy as np, time

# ---------------------------
# 1) Define your zones here
#    Coordinates are in IMAGE PIXELS of the input frame.
#    Example below assumes ~1024x768-ish frames; adjust points to your camera view.
# ---------------------------
ZONES = [
    {
        "name": "Gate Area",
        "poly": [(100,120),(940,120),(940,420),(100,420)],
        "thr": {"person": 30}   # alert when ≥30 persons in this zone
    },
    {
        "name": "Corridor",
        "poly": [(140,460),(900,460),(900,740),(140,740)],
        "thr": {"person": 15}
    }
]

# color per zone for drawing (optional)
ZONE_COLORS = [(0,255,255),(255,128,0),(128,255,0),(255,0,128)]

# ---------------------------
# 2) Density at frame size (sum-preserving)
#    Uses your trained density_net via density_count_bgr(), then upsamples to frame with sum preserved.
# ---------------------------
def density_map_at_frame(frame_bgr):
    """
    Returns dm_full (H,W) and count where dm_full.sum() == count (sum-preserving).
    """
    dm_small, c_den = density_count_bgr(frame_bgr)    # from Step 5; dm_small is (hp,wp)
    hp, wp = dm_small.shape[:2]
    H, W = frame_bgr.shape[:2]
    # bilinear upsample + area correction so the sum stays invariant
    dm_full = cv2.resize(dm_small.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    dm_full *= (hp * wp) / float(H * W)               # preserve integral after resizing
    # numerical drift fix: align final sum to c_den
    if dm_full.sum() > 0:
        dm_full *= (c_den / dm_full.sum())
    return dm_full, float(c_den)

# ---------------------------
# 3) Helpers: polygon masks & zone counts
# ---------------------------
def polygon_mask(h, w, poly):
    mask = np.zeros((h,w), dtype=np.uint8)
    pts = np.array(poly, dtype=np.int32)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)

def zone_counts(frame_bgr, dets, counts_det):
    """
    Returns:
      zone_people: dict zone->fused person count (float)
      zone_classes: dict zone->{class->int} (from YOLO)
    """
    H, W = frame_bgr.shape[:2]
    # YOLO persons per zone (by center point)
    zone_classes = {z["name"]: {k:0 for k in INTEREST_CLASSES} for z in ZONES}
    for d in dets:
        cx, cy = d["center"]
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        for z in ZONES:
            if cv2.pointPolygonTest(np.array(z["poly"], np.int32), (float(cx), float(cy)), False) >= 0:
                cname = d["cls"]
                if cname in zone_classes[z["name"]]:
                    zone_classes[z["name"]][cname] += 1

    # Density per zone (sum inside polygon)
    dm_full, c_den_global = density_map_at_frame(frame_bgr)
    zone_people_density = {}
    for z in ZONES:
        mask = polygon_mask(H, W, z["poly"])
        zone_people_density[z["name"]] = float(dm_full[mask].sum())

    # Fused per zone: prefer density when crowded or when density >> YOLO
    zone_people_fused = {}
    for z in ZONES:
        y = float(zone_classes[z["name"]].get("person", 0))
        d = float(zone_people_density[z["name"]])
        use_density = (d >= max(DENSITY_MIN_VALID, SWITCH_TO_DENSITY)) or (d > y * DENSITY_DOMINATES_FACTOR)
        zone_people_fused[z["name"]] = (d if use_density else y)
    return zone_people_fused, zone_classes

# ---------------------------
# 4) Draw zones and HUD
# ---------------------------
def draw_zones(vis):
    for i, z in enumerate(ZONES):
        col = ZONE_COLORS[i % len(ZONE_COLORS)]
        pts = np.array(z["poly"], dtype=np.int32)
        cv2.polylines(vis, [pts], True, col, 2)
        # label
        cx = int(np.mean([p[0] for p in z["poly"]]))
        cy = int(np.mean([p[1] for p in z["poly"]]))
        cv2.putText(vis, z["name"], (cx-40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)

# ---------------------------
# 5) Single-image with zones (analysis + alerts)
# ---------------------------
from matplotlib import pyplot as plt
from google.colab import files

def analyze_image_with_zones(path=None, trigger_alerts=True, show=True):
    # pick image
    if path is None:
        up = files.upload()
        if not up:
            print("No file uploaded.");
            return
        path = list(up.keys())[0]
    img = cv2.imread(path); assert img is not None, f"failed to read {path}"
    vis = img.copy()

    # YOLO
    dets, counts_det = run_yolo(img)

    # draw YOLO boxes
    for d in dets:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis, f"{d['cls']} {d['conf']:.2f}", (x1,y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    # Global fused
    fused_global, src = fused_people_count(counts_det.get("person",0), img)
    cv2.putText(vis, f"Global persons (fused-{src}): {fused_global:.1f}", (12,30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)

    # Zones
    draw_zones(vis)
    zone_people, zone_classes = zone_counts(img, dets, counts_det)

    y = 60
    for z in ZONES:
        name = z["name"]; val = zone_people[name]
        cv2.putText(vis, f"{name}: {val:.1f} persons", (12,y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2); y+=26

    # Alerts (zone-level + global animals)
    if trigger_alerts:
        # per-zone person thresholds
        for z in ZONES:
            thr = z.get("thr",{}).get("person", None)
            if thr is not None and zone_people[z["name"]] >= thr and should_alert(f"ZONE_{z['name']}_person"):
                msg = f"🚨 Zone Alert [{z['name']}]: persons={zone_people[z['name']]:.1f} (≥{thr})"
                send_telegram(msg, photo_bgr=vis)
                publish_mqtt("alerts", {"type":"zone_person_threshold","zone":z["name"],"value":float(zone_people[z['name']]),"thr":thr,"ts":time.time()})

        # animal-in-crowd (global)
        animals = [k for k,v in counts_det.items() if k in ANIMAL_CLASSES and v>0]
        if ALERT_ON_ANIMAL_IN_CROWD and fused_global>=1 and animals and should_alert("IMG_ZONE_animal"):
            msg = f"⚠️ Animal(s) in crowd: {', '.join(animals)} | persons≈{fused_global:.1f}"
            send_telegram(msg, photo_bgr=vis)
            publish_mqtt("alerts", {"type":"img_animal_in_crowd","animals":animals,"people":float(fused_global),"ts":time.time()})

    # show
    if show:
        plt.figure(figsize=(12,7))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

    print("Global fused persons:", f"{fused_global:.1f}")
    print("Zone persons (fused):", {z['name']: round(zone_people[z['name']],1) for z in ZONES})
    for z in ZONES:
        nz = {k:v for k,v in zone_classes[z["name"]].items() if v>0}
        if nz:
            print(f"Zone '{z['name']}' classes:", nz)
    return vis, zone_people, zone_classes

# ---------------------------
# 6) Video/RTSP with zones
# ---------------------------
from google.colab.patches import cv2_imshow

def process_stream_with_zones(source, show_every=3, max_frames=None):
    cap = cv2.VideoCapture(source)
    assert cap.isOpened(), f"Could not open source: {source}"
    n=0; alert_hits=0
    t0=time.time()

    while True:
        ok, frame = cap.read()
        if not ok: break
        n+=1
        vis = frame.copy()

        dets, counts_det = run_yolo(frame)
        for d in dets:
            x1,y1,x2,y2 = d["bbox"]
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)

        fused_global, src = fused_people_count(counts_det.get("person",0), frame)
        cv2.putText(vis, f"Global persons (fused-{src}): {fused_global:.1f}", (12,30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)

        draw_zones(vis)
        zone_people, zone_classes = zone_counts(frame, dets, counts_det)

        y = 60
        for z in ZONES:
            name=z["name"]; val=zone_people[name]
            cv2.putText(vis, f"{name}: {val:.1f} persons", (12,y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2); y+=26

        # alerts per-zone
        for z in ZONES:
            thr = z.get("thr",{}).get("person", None)
            if thr is not None and zone_people[z["name"]] >= thr and should_alert(f"VID_ZONE_{z['name']}_person"):
                alert_hits+=1
                msg=f"🚨 Zone Alert [{z['name']}]: persons={zone_people[z['name']]:.1f} (≥{thr})"
                send_telegram(msg, photo_bgr=vis)
                publish_mqtt("alerts", {"type":"video_zone_person_threshold","zone":z["name"],"value":float(zone_people[z['name']]),"thr":thr,"ts":time.time()})

        # animal-in-crowd (global)
        animals = [k for k,v in counts_det.items() if k in ANIMAL_CLASSES and v>0]
        if ALERT_ON_ANIMAL_IN_CROWD and fused_global>=1 and animals and should_alert("VID_ZONE_animal"):
            alert_hits+=1
            msg=f"⚠️ Animal(s) in crowd: {', '.join(animals)} | persons≈{fused_global:.1f}"
            send_telegram(msg, photo_bgr=vis)
            publish_mqtt("alerts", {"type":"video_animal_in_crowd","animals":animals,"people":float(fused_global),"ts":time.time()})

        if (n % show_every) == 0:
            cv2_imshow(vis)
        if max_frames and n >= max_frames:
            break

    cap.release()
    dt=time.time()-t0
    print(f"[done] frames={n} | alerts sent={alert_hits} | {dt:.1f}s")


In [ ]:
# STEP 8 — Interactive Zone Editor (matplotlib)
# Click to draw polygons, name them, set thresholds, save to JSON -> updates global ZONES

import json, os, cv2, numpy as np
from matplotlib import pyplot as plt
from google.colab import files

ZONES = []  # we will overwrite this with what you draw

def _to_rgb(img_bgr):  # helper for display
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def start_zone_editor(image_path=None, save_path="zones.json"):
    """
    Click to add vertices.
      Keys:
        Enter/c  : close/save current polygon (will prompt for name and person-threshold)
        z        : undo last point
        n        : start a new polygon (discard current unfinished)
        s        : save all polygons to JSON and update global ZONES
        q        : quit editor
    """
    global ZONES
    if image_path is None:
        up = files.upload()
        if not up:
            print("No file uploaded.");
            return
        image_path = list(up.keys())[0]

    bg = cv2.imread(image_path)
    assert bg is not None, f"failed to read {image_path}"
    H, W = bg.shape[:2]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(_to_rgb(bg))
    ax.set_title("Click to add points. Enter/c=close, z=undo, n=new, s=save, q=quit")
    pts = []
    dots = []
    lines = []
    preview_poly = None

    def redraw_preview():
        nonlocal preview_poly
        if preview_poly:
            preview_poly.remove()
            preview_poly = None
        if len(pts) >= 2:
            xs = [p[0] for p in pts] + [pts[-1][0]]
            ys = [p[1] for p in pts] + [pts[-1][1]]
            preview_poly, = ax.plot(xs, ys, '-', color='yellow', lw=2)
        fig.canvas.draw_idle()

    def onclick(event):
        if not event.inaxes:
            return
        if event.button == 1:  # left click
            x, y = int(event.xdata), int(event.ydata)
            pts.append((x, y))
            dots.append(ax.plot([x], [y], 'ro', ms=5)[0])
            redraw_preview()

    def onkey(event):
        nonlocal pts, dots, lines, preview_poly
        key = event.key
        if key == 'z':  # undo point
            if pts:
                pts.pop()
                h = dots.pop()
                h.remove()
                redraw_preview()
                fig.canvas.draw_idle()
        elif key in ('enter', 'c'):  # close/save polygon
            if len(pts) >= 3:
                # draw closed polygon
                xs = [p[0] for p in pts] + [pts[0][0]]
                ys = [p[1] for p in pts] + [pts[0][1]]
                lines.append(ax.plot(xs, ys, '-', color='cyan', lw=2)[0])
                # prompt for name and threshold (in notebook input)
                name = input("Zone name: ").strip() or f"Zone{len(ZONES)+1}"
                try:
                    thr = float(input("Person threshold for alert (e.g., 30): ").strip() or "0")
                except:
                    thr = 0.0
                ZONES.append({"name": name, "poly": pts.copy(), "thr": {"person": thr}})
                # clear current points
                for d in dots: d.remove()
                dots.clear()
                pts.clear()
                redraw_preview()
                fig.canvas.draw_idle()
                print(f"[saved] {name} with {len(ZONES[-1]['poly'])} pts, thr={thr}")
            else:
                print("Need at least 3 points to close a polygon.")
        elif key == 'n':  # new polygon (discard current)
            for d in dots: d.remove()
            dots.clear()
            pts.clear()
            redraw_preview()
            fig.canvas.draw_idle()
            print("[info] started a new polygon.")
        elif key == 's':  # save to JSON
            with open(save_path, "w") as f:
                json.dump(ZONES, f, indent=2)
            print(f"[saved] {save_path}")
        elif key == 'q':  # quit
            plt.close(fig)

    cid1 = fig.canvas.mpl_connect('button_press_event', onclick)
    cid2 = fig.canvas.mpl_connect('key_press_event', onkey)
    plt.show()

    # After window closed:
    # If zones.json exists, re-load and overwrite global ZONES to ensure consistency
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            ZONES = json.load(f)
        print(f"[loaded] {save_path} -> ZONES ({len(ZONES)} zones)")
    else:
        print("[info] no zones saved this session.")

# --- usage ---
# start_zone_editor("/content/ShanghaiTech/ShanghaiTechB/test_data/images/IMG_29.jpg")
# or simply:
# start_zone_editor()


# Configurable User interface for custom images and videos input

In [ ]:
# ==== RETUNE PATCH + CONFIGURABLE COLAB UI (uses your trained models) ====
# Assumes these are already defined from Steps 5–7:
# yolo, density_net, density_count_bgr, run_yolo, fused_people_count,
# send_telegram, publish_mqtt, should_alert, INTEREST_CLASSES, ANIMAL_CLASSES, (optional) ZONES

import os, cv2, json, math, time, traceback, numpy as np
from IPython.display import display, clear_output, HTML
from google.colab import files
import ipywidgets as w

# ---- 0) Reload your new CSRNet weights (safe if already loaded) ----
CSRNET_WEIGHTS = "/content/weights_csrnet_lite_stb_best.pt"
try:
    if 'density_net' in globals() and os.path.exists(CSRNET_WEIGHTS):
        density_net.load_state_dict(torch.load(CSRNET_WEIGHTS, map_location=next(density_net.parameters()).device))
        density_net.eval()
        print(f"[init] Reloaded CSRNet weights → {CSRNET_WEIGHTS}")
    else:
        print("[init] CSRNet weights not found or model missing; continuing with current state.")
except Exception as e:
    print("[init] reload failed:", e)

# ---- 1) Helpers (draw + density) ----
def _draw_all(vis, dets, counts_det, fused_global, fused_src):
    vis = vis.copy()
    for d in dets:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)
    cv2.putText(vis,f"Persons (fused-{fused_src}): {fused_global:.1f}",(12,28),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2)
    return vis

def _polygon_mask(h, w, poly):
    m = np.zeros((h,w), dtype=np.uint8)
    pts = np.array(poly, dtype=np.int32)
    cv2.fillPoly(m, [pts], 1)
    return m.astype(bool)

def density_map_at_frame(frame_bgr):
    """Sum-preserving upsample of your density map to full frame."""
    dm_small, c_den = density_count_bgr(frame_bgr)
    hp, wp = dm_small.shape[:2]
    H, W = frame_bgr.shape[:2]
    dm_full = cv2.resize(dm_small.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    dm_full *= (hp * wp) / float(H * W)
    s = dm_full.sum()
    if s > 0:
        dm_full *= (c_den / s)
    return dm_full, float(c_den)

def density_sum_box_masked(img_bgr, dets, expand_ratio=0.15, require_person=True):
    """
    Sum density only inside YOLO 'person' boxes (expanded). If require_person and no boxes, return 0.
    """
    H, W = img_bgr.shape[:2]
    dm_full, _ = density_map_at_frame(img_bgr)
    mask = np.zeros((H,W), dtype=np.uint8)
    person_boxes = [d["bbox"] for d in dets if d["cls"] == "person"]
    if require_person and not person_boxes:
        return 0.0
    for (x1,y1,x2,y2) in person_boxes:
        dx = int((x2-x1) * expand_ratio); dy = int((y2-y1) * expand_ratio)
        x1e = max(0, x1-dx); y1e = max(0, y1-dy)
        x2e = min(W-1, x2+dx); y2e = min(H-1, y2+dy)
        mask[y1e:y2e+1, x1e:x2e+1] = 1
    return float(dm_full[mask.astype(bool)].sum())

def zone_counts(frame_bgr, dets, counts_det,
                switch_to_density=10.0, dom_factor=1.2, min_valid=5.0,
                use_mask=True, expand_ratio=0.15):
    """Zone-wise fused counts with optional density masking."""
    if 'ZONES' not in globals() or not ZONES:
        return {}, {}
    H, W = frame_bgr.shape[:2]

    # YOLO per zone
    zone_classes = {z["name"]: {k:0 for k in INTEREST_CLASSES} for z in ZONES}
    for d in dets:
        cx, cy = d["center"]
        if not (0 <= cx < W and 0 <= cy < H): continue
        for z in ZONES:
            pts = np.array(z["poly"], np.int32)
            if cv2.pointPolygonTest(pts, (float(cx), float(cy)), False) >= 0:
                cname = d["cls"]
                if cname in zone_classes[z["name"]]:
                    zone_classes[z["name"]][cname] += 1

    # Density per zone
    dm_full, _ = density_map_at_frame(frame_bgr)
    zone_people_density = {}
    for z in ZONES:
        mask = _polygon_mask(H, W, z["poly"])
        if use_mask:
            # combine with person boxes
            m2 = np.zeros((H,W), dtype=np.uint8)
            for d in dets:
                if d["cls"] != "person": continue
                x1,y1,x2,y2 = d["bbox"]
                m2[y1:y2+1, x1:x2+1] = 1
            zmask = (mask & m2.astype(bool))
        else:
            zmask = mask
        zone_people_density[z["name"]] = float(dm_full[zmask].sum())

    # Fused per zone
    zone_people_fused={}
    for z in ZONES:
        y = float(zone_classes[z["name"]].get("person", 0))
        d = float(zone_people_density[z["name"]])
        use_den = (d >= max(min_valid, switch_to_density)) or (d > y * dom_factor)
        zone_people_fused[z["name"]] = (d if use_den else y)
    return zone_people_fused, zone_classes

def draw_zones(vis, zones_people):
    if 'ZONES' not in globals() or not ZONES: return vis
    vis = vis.copy()
    for i, z in enumerate(ZONES):
        color = (0,255,255) if i%2==0 else (255,128,0)
        pts = np.array(z["poly"], dtype=np.int32)
        cv2.polylines(vis, [pts], True, color, 2)
        cx = int(np.mean([p[0] for p in z["poly"]])); cy = int(np.mean([p[1] for p in z["poly"]]))
        cv2.putText(vis, f"{z['name']}: {zones_people.get(z['name'],0):.1f}", (cx-60, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return vis

# ---- 2) ipywidgets UI with retuned defaults (density is stronger now) ----
hdr = w.HTML("<h3>Crowd Count · YOLOv12 + Density Fusion · Retuned Interface</h3>")

# Fusion/alert defaults tuned for your 20-epoch model
person_thr = w.IntSlider(value=50, min=1, max=500, step=1, description="Global threshold")
animal_flag = w.Checkbox(value=True, description="Alert if animal in crowd")

switch_to_density = w.FloatSlider(value=8.0,  min=0, max=150, step=0.5, description="Switch@≥ density")
dom_factor       = w.FloatSlider(value=1.15, min=1.0, max=3.0, step=0.05, description="Density > YOLO ×")
min_valid        = w.FloatSlider(value=4.0,  min=0, max=50,  step=0.5, description="Min valid density")

mask_density   = w.Checkbox(value=True, description="Mask density to YOLO person boxes")
expand_percent = w.FloatSlider(value=15.0, min=0, max=50, step=1.0, description="Box expand %")

# External alerts (optional)
enable_external = w.Checkbox(value=True, description="Enable external alerts (Telegram/MQTT)")
tg_token = w.Text(value=globals().get("BOT_TOKEN",""), description="Telegram BOT_TOKEN", layout=w.Layout(width="60%"))
tg_chat  = w.Text(value=str(globals().get("CHAT_ID","")), description="Telegram CHAT_ID", layout=w.Layout(width="40%"))
mqtt_host = w.Text(value=globals().get("MQTT_HOST","broker.emqx.io"), description="MQTT host", layout=w.Layout(width="45%"))
mqtt_port = w.IntText(value=globals().get("MQTT_PORT",1883), description="Port", layout=w.Layout(width="20%"))
mqtt_topic= w.Text(value=globals().get("MQTT_TOPIC_BASE","site/demo/camera/colab"), description="MQTT topic base", layout=w.Layout(width="35%"))

# Zones
ZONES = globals().get("ZONES", [])
zones_status = w.HTML(value=f"<b>Zones:</b> {'Loaded' if ZONES else 'None'}")
def _load_zones(_):
    global ZONES
    if os.path.exists("zones.json"):
        try:
            with open("zones.json","r") as f: ZONES = json.load(f)
            zones_status.value = f"<b>Zones:</b> Loaded {len(ZONES)}"
        except Exception as e:
            zones_status.value = f"<b>Zones:</b> read error ({e})"
    else:
        zones_status.value = "<b>Zones:</b> zones.json not found"
btn_load_zones = w.Button(description="Load zones.json"); btn_load_zones.on_click(_load_zones)

# Reload weights button
w_path = w.Text(value=CSRNET_WEIGHTS, description="CSRNet weights", layout=w.Layout(width="70%"))
def _reload(_):
    try:
        density_net.load_state_dict(torch.load(w_path.value, map_location=next(density_net.parameters()).device))
        density_net.eval(); print(f"[ok] reloaded {w_path.value}")
    except Exception as e:
        print("[reload err]", e)
btn_reload = w.Button(description="Reload weights", button_style="")

# Output
out = w.Output()

# Utils to wire external creds
def _apply_external_settings():
    global BOT_TOKEN, CHAT_ID, MQTT_HOST, MQTT_PORT, MQTT_TOPIC_BASE
    BOT_TOKEN = tg_token.value.strip()
    try: CHAT_ID = int(tg_chat.value.strip()) if tg_chat.value.strip() else 0
    except: CHAT_ID = 0
    MQTT_HOST = mqtt_host.value.strip()
    MQTT_PORT = int(mqtt_port.value)
    MQTT_TOPIC_BASE = mqtt_topic.value.strip()

def _toggle_alert_shims(enable: bool):
    """No-op the senders when disabled."""
    global _send_real, _publish_real
    if enable:
        if "_send_real" in globals(): globals()["send_telegram"] = _send_real
        if "_publish_real" in globals(): globals()["publish_mqtt"] = _publish_real
    else:
        if "send_telegram" in globals() and "_send_real" not in globals():
            _send_real = globals()["send_telegram"]
        if "publish_mqtt" in globals() and "_publish_real" not in globals():
            _publish_real = globals()["publish_mqtt"]
        globals()["send_telegram"] = lambda *a, **k: None
        globals()["publish_mqtt"] = lambda *a, **k: None

# ---- 3) Image runner ----
btn_img = w.Button(description="Upload & Run (Images)", button_style="primary", icon="upload")

def _on_images(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No images uploaded."); return
            for name, data in up.items():
                try:
                    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
                    assert img is not None, f"decode failed: {name}"
                    dets, counts = run_yolo(img)

                    # density (masked or raw)
                    if mask_density.value:
                        d_local = density_sum_box_masked(img, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                    else:
                        _, d_local_raw = density_count_bgr(img)
                        d_local = float(d_local_raw)

                    y_local = float(counts.get("person",0))
                    animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                    use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)

                    # animal safeguard – prefer YOLO when animals are around and people exist
                    if animals_present and y_local > 0:
                        use_den = False

                    fused = (d_local if use_den else y_local)
                    src = "density" if use_den else "yolo"

                    vis = _draw_all(img, dets, counts, fused, src)

                    zones_people = {}
                    if ZONES:
                        zones_people, _ = zone_counts(img, dets, counts,
                                                      switch_to_density.value, dom_factor.value, min_valid.value,
                                                      use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                        vis = draw_zones(vis, zones_people)

                    # alerts
                    msgs=[]
                    if fused >= person_thr.value and should_alert("RETUNE_IMG"):
                        m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                        msgs.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"img_person_threshold","value":float(fused),"thr":person_thr.value,"ts":time.time()})
                    if animal_flag.value and fused>=1:
                        animals=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                        if animals and should_alert("RETUNE_IMG_ANIMAL"):
                            m=f"⚠️ Animal(s) in crowd: {', '.join(animals)} | persons≈{fused:.1f}"
                            msgs.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"img_animal_in_crowd","animals":animals,"people":float(fused),"ts":time.time()})

                    os.makedirs("/content/outputs/images", exist_ok=True)
                    out_path = f"/content/outputs/images/ANN_{os.path.basename(name)}"
                    cv2.imwrite(out_path, vis)

                    print(f"✅ {name} → {out_path}")
                    print(json.dumps({
                        "file": name,
                        "fused_persons": round(float(fused),1),
                        "fused_source": src,
                        "yolo_counts": {k:v for k,v in counts.items() if v>0},
                        "zones_persons": {k: round(v,1) for k,v in zones_people.items()} if zones_people else {},
                        **({"alerts": msgs} if msgs else {})
                    }, indent=2))
                    cv2_imshow(vis)
                except Exception as e:
                    print("[image error]", name, e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_img.on_click(_on_images)

# ---- 4) Video runner ----
btn_vid = w.Button(description="Upload & Run (Video)", button_style="primary", icon="upload")
show_every = w.IntSlider(value=6, min=1, max=30, step=1, description="Preview every N frames")
max_frames = w.IntText(value=0, description="Max frames (0=all)")
downscale_pixels = w.IntText(value=1280*720, description="Max pixels (downscale)")

def _on_video(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No video uploaded."); return
            vname = next(iter(up.keys()))
            vpath = f"/content/{vname}"; open(vpath,"wb").write(up[vname])

            cap = cv2.VideoCapture(vpath)
            assert cap.isOpened(), f"open failed: {vname}"
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

            scale = min(1.0, math.sqrt(downscale_pixels.value / max(1, W*H)))
            W2, H2 = max(64, int(W*scale)), max(64, int(H*scale))

            os.makedirs("/content/outputs/videos", exist_ok=True)
            out_path = f"/content/outputs/videos/processed_{int(time.time())}.mp4"
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W2,H2))

            n=0; alerts=[]
            t0=time.time()
            print(f"[run] {vname} → {out_path} | {W}x{H}@{fps:.1f} -> {W2}x{H2}")

            while True:
                ok, frame = cap.read()
                if not ok: break
                n+=1
                if max_frames.value and n>int(max_frames.value): break
                if scale < 0.999:
                    frame = cv2.resize(frame,(W2,H2), interpolation=cv2.INTER_AREA)

                dets, counts = run_yolo(frame)

                # density masked or raw
                if mask_density.value:
                    d_local = density_sum_box_masked(frame, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                else:
                    _, d_raw = density_count_bgr(frame); d_local=float(d_raw)

                y_local = float(counts.get("person",0))
                animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)
                if animals_present and y_local > 0:
                    use_den = False

                fused = (d_local if use_den else y_local)
                src = "density" if use_den else "yolo"

                vis = _draw_all(frame, dets, counts, fused, src)

                if 'ZONES' in globals() and ZONES:
                    zones_people, _ = zone_counts(frame, dets, counts,
                                                  switch_to_density.value, dom_factor.value, min_valid.value,
                                                  use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                    vis = draw_zones(vis, zones_people)
                    for z in ZONES:
                        thr = z.get("thr",{}).get("person", None)
                        if thr is not None and zones_people.get(z["name"],0) >= thr and should_alert(f"RETUNE_VID_ZONE_{z['name']}"):
                            m=f"🚨 Zone Alert [{z['name']}]: persons={zones_people[z['name']]:.1f} (≥{thr})"
                            alerts.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"video_zone_person_threshold","zone":z["name"],
                                                    "value":float(zones_people[z['name']]),"thr":thr,"ts":time.time()})

                # global alerts
                if fused >= person_thr.value and should_alert("RETUNE_VID"):
                    m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                    alerts.append(m); send_telegram(m, photo_bgr=vis)
                    publish_mqtt("alerts", {"type":"video_person_threshold","value":float(fused),
                                            "thr":person_thr.value,"ts":time.time()})
                if animal_flag.value and fused>=1:
                    a=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                    if a and should_alert("RETUNE_VID_ANIMAL"):
                        m=f"⚠️ Animal(s) in crowd: {', '.join(a)} | persons≈{fused:.1f}"
                        alerts.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"video_animal_in_crowd","animals":a,
                                                "people":float(fused),"ts":time.time()})

                if (n % max(1, int(show_every.value))) == 0:
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(vis)

                writer.write(vis)

            writer.release(); cap.release()
            dt=time.time()-t0
            print(f"\n[done] frames={n} | alerts={len(alerts)} | saved → {out_path} | {dt:.1f}s")
            print(json.dumps({"frames":n,"alerts":alerts,"output":out_path}, indent=2))
            try:
                display(HTML(f"""<video width="720" controls>
                  <source src="file://{out_path}" type="video/mp4"></video>"""))
            except: pass

        except Exception as e:
            print("[video error]", e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_vid.on_click(_on_video)

# ---- 5) Layout & display ----
row1 = w.HBox([person_thr, animal_flag])
row2 = w.HBox([switch_to_density, dom_factor, min_valid])
row3 = w.HBox([mask_density, expand_percent])
row4 = w.HBox([enable_external])
row5 = w.HBox([tg_token, tg_chat])
row6 = w.HBox([mqtt_host, mqtt_port, mqtt_topic])
row7 = w.HBox([w_path, btn_reload])
rowZ = w.HBox([btn_load_zones, zones_status])

tab_images = w.VBox([w.HTML("<b>Images</b>"), btn_img])
tab_video  = w.VBox([w.HTML("<b>Video</b>"), btn_vid, show_every, max_frames, downscale_pixels])

tabs = w.Tab(children=[tab_images, tab_video])
tabs.set_title(0, "Images"); tabs.set_title(1, "Video")

ui = w.VBox([hdr, row1, row2, row3, row4, row5, row6, rowZ, row7, tabs, out])
display(ui)


Video Test Run

In [ ]:
# ==== RETUNE PATCH + CONFIGURABLE COLAB UI (uses your trained models) ====
# Assumes these are already defined from Steps 5–7:
# yolo, density_net, density_count_bgr, run_yolo, fused_people_count,
# send_telegram, publish_mqtt, should_alert, INTEREST_CLASSES, ANIMAL_CLASSES, (optional) ZONES

import os, cv2, json, math, time, traceback, numpy as np
from IPython.display import display, clear_output, HTML
from google.colab import files
import ipywidgets as w

# ---- 0) Reload your new CSRNet weights (safe if already loaded) ----
CSRNET_WEIGHTS = "/content/weights_csrnet_lite_stb_best.pt"
try:
    if 'density_net' in globals() and os.path.exists(CSRNET_WEIGHTS):
        density_net.load_state_dict(torch.load(CSRNET_WEIGHTS, map_location=next(density_net.parameters()).device))
        density_net.eval()
        print(f"[init] Reloaded CSRNet weights → {CSRNET_WEIGHTS}")
    else:
        print("[init] CSRNet weights not found or model missing; continuing with current state.")
except Exception as e:
    print("[init] reload failed:", e)

# ---- 1) Helpers (draw + density) ----
def _draw_all(vis, dets, counts_det, fused_global, fused_src):
    vis = vis.copy()
    for d in dets:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)
    cv2.putText(vis,f"Persons (fused-{fused_src}): {fused_global:.1f}",(12,28),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2)
    return vis

def _polygon_mask(h, w, poly):
    m = np.zeros((h,w), dtype=np.uint8)
    pts = np.array(poly, dtype=np.int32)
    cv2.fillPoly(m, [pts], 1)
    return m.astype(bool)

def density_map_at_frame(frame_bgr):
    """Sum-preserving upsample of your density map to full frame."""
    dm_small, c_den = density_count_bgr(frame_bgr)
    hp, wp = dm_small.shape[:2]
    H, W = frame_bgr.shape[:2]
    dm_full = cv2.resize(dm_small.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    dm_full *= (hp * wp) / float(H * W)
    s = dm_full.sum()
    if s > 0:
        dm_full *= (c_den / s)
    return dm_full, float(c_den)

def density_sum_box_masked(img_bgr, dets, expand_ratio=0.15, require_person=True):
    """
    Sum density only inside YOLO 'person' boxes (expanded). If require_person and no boxes, return 0.
    """
    H, W = img_bgr.shape[:2]
    dm_full, _ = density_map_at_frame(img_bgr)
    mask = np.zeros((H,W), dtype=np.uint8)
    person_boxes = [d["bbox"] for d in dets if d["cls"] == "person"]
    if require_person and not person_boxes:
        return 0.0
    for (x1,y1,x2,y2) in person_boxes:
        dx = int((x2-x1) * expand_ratio); dy = int((y2-y1) * expand_ratio)
        x1e = max(0, x1-dx); y1e = max(0, y1-dy)
        x2e = min(W-1, x2+dx); y2e = min(H-1, y2+dy)
        mask[y1e:y2e+1, x1e:x2e+1] = 1
    return float(dm_full[mask.astype(bool)].sum())

def zone_counts(frame_bgr, dets, counts_det,
                switch_to_density=10.0, dom_factor=1.2, min_valid=5.0,
                use_mask=True, expand_ratio=0.15):
    """Zone-wise fused counts with optional density masking."""
    if 'ZONES' not in globals() or not ZONES:
        return {}, {}
    H, W = frame_bgr.shape[:2]

    # YOLO per zone
    zone_classes = {z["name"]: {k:0 for k in INTEREST_CLASSES} for z in ZONES}
    for d in dets:
        cx, cy = d["center"]
        if not (0 <= cx < W and 0 <= cy < H): continue
        for z in ZONES:
            pts = np.array(z["poly"], np.int32)
            if cv2.pointPolygonTest(pts, (float(cx), float(cy)), False) >= 0:
                cname = d["cls"]
                if cname in zone_classes[z["name"]]:
                    zone_classes[z["name"]][cname] += 1

    # Density per zone
    dm_full, _ = density_map_at_frame(frame_bgr)
    zone_people_density = {}
    for z in ZONES:
        mask = _polygon_mask(H, W, z["poly"])
        if use_mask:
            # combine with person boxes
            m2 = np.zeros((H,W), dtype=np.uint8)
            for d in dets:
                if d["cls"] != "person": continue
                x1,y1,x2,y2 = d["bbox"]
                m2[y1:y2+1, x1:x2+1] = 1
            zmask = (mask & m2.astype(bool))
        else:
            zmask = mask
        zone_people_density[z["name"]] = float(dm_full[zmask].sum())

    # Fused per zone
    zone_people_fused={}
    for z in ZONES:
        y = float(zone_classes[z["name"]].get("person", 0))
        d = float(zone_people_density[z["name"]])
        use_den = (d >= max(min_valid, switch_to_density)) or (d > y * dom_factor)
        zone_people_fused[z["name"]] = (d if use_den else y)
    return zone_people_fused, zone_classes

def draw_zones(vis, zones_people):
    if 'ZONES' not in globals() or not ZONES: return vis
    vis = vis.copy()
    for i, z in enumerate(ZONES):
        color = (0,255,255) if i%2==0 else (255,128,0)
        pts = np.array(z["poly"], dtype=np.int32)
        cv2.polylines(vis, [pts], True, color, 2)
        cx = int(np.mean([p[0] for p in z["poly"]])); cy = int(np.mean([p[1] for p in z["poly"]]))
        cv2.putText(vis, f"{z['name']}: {zones_people.get(z['name'],0):.1f}", (cx-60, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return vis

# ---- 2) ipywidgets UI with retuned defaults (density is stronger now) ----
hdr = w.HTML("<h3>Crowd Count · YOLOv12 + Density Fusion · Retuned Interface</h3>")

# Fusion/alert defaults tuned for your 20-epoch model
person_thr = w.IntSlider(value=50, min=1, max=500, step=1, description="Global threshold")
animal_flag = w.Checkbox(value=True, description="Alert if animal in crowd")

switch_to_density = w.FloatSlider(value=8.0,  min=0, max=150, step=0.5, description="Switch@≥ density")
dom_factor       = w.FloatSlider(value=1.15, min=1.0, max=3.0, step=0.05, description="Density > YOLO ×")
min_valid        = w.FloatSlider(value=4.0,  min=0, max=50,  step=0.5, description="Min valid density")

mask_density   = w.Checkbox(value=True, description="Mask density to YOLO person boxes")
expand_percent = w.FloatSlider(value=15.0, min=0, max=50, step=1.0, description="Box expand %")

# External alerts (optional)
enable_external = w.Checkbox(value=True, description="Enable external alerts (Telegram/MQTT)")
tg_token = w.Text(value=globals().get("BOT_TOKEN",""), description="Telegram BOT_TOKEN", layout=w.Layout(width="60%"))
tg_chat  = w.Text(value=str(globals().get("CHAT_ID","")), description="Telegram CHAT_ID", layout=w.Layout(width="40%"))
mqtt_host = w.Text(value=globals().get("MQTT_HOST","broker.emqx.io"), description="MQTT host", layout=w.Layout(width="45%"))
mqtt_port = w.IntText(value=globals().get("MQTT_PORT",1883), description="Port", layout=w.Layout(width="20%"))
mqtt_topic= w.Text(value=globals().get("MQTT_TOPIC_BASE","site/demo/camera/colab"), description="MQTT topic base", layout=w.Layout(width="35%"))

# Zones
ZONES = globals().get("ZONES", [])
zones_status = w.HTML(value=f"<b>Zones:</b> {'Loaded' if ZONES else 'None'}")
def _load_zones(_):
    global ZONES
    if os.path.exists("zones.json"):
        try:
            with open("zones.json","r") as f: ZONES = json.load(f)
            zones_status.value = f"<b>Zones:</b> Loaded {len(ZONES)}"
        except Exception as e:
            zones_status.value = f"<b>Zones:</b> read error ({e})"
    else:
        zones_status.value = "<b>Zones:</b> zones.json not found"
btn_load_zones = w.Button(description="Load zones.json"); btn_load_zones.on_click(_load_zones)

# Reload weights button
w_path = w.Text(value=CSRNET_WEIGHTS, description="CSRNet weights", layout=w.Layout(width="70%"))
def _reload(_):
    try:
        density_net.load_state_dict(torch.load(w_path.value, map_location=next(density_net.parameters()).device))
        density_net.eval(); print(f"[ok] reloaded {w_path.value}")
    except Exception as e:
        print("[reload err]", e)
btn_reload = w.Button(description="Reload weights", button_style="")

# Output
out = w.Output()

# Utils to wire external creds
def _apply_external_settings():
    global BOT_TOKEN, CHAT_ID, MQTT_HOST, MQTT_PORT, MQTT_TOPIC_BASE
    BOT_TOKEN = tg_token.value.strip()
    try: CHAT_ID = int(tg_chat.value.strip()) if tg_chat.value.strip() else 0
    except: CHAT_ID = 0
    MQTT_HOST = mqtt_host.value.strip()
    MQTT_PORT = int(mqtt_port.value)
    MQTT_TOPIC_BASE = mqtt_topic.value.strip()

def _toggle_alert_shims(enable: bool):
    """No-op the senders when disabled."""
    global _send_real, _publish_real
    if enable:
        if "_send_real" in globals(): globals()["send_telegram"] = _send_real
        if "_publish_real" in globals(): globals()["publish_mqtt"] = _publish_real
    else:
        if "send_telegram" in globals() and "_send_real" not in globals():
            _send_real = globals()["send_telegram"]
        if "publish_mqtt" in globals() and "_publish_real" not in globals():
            _publish_real = globals()["publish_mqtt"]
        globals()["send_telegram"] = lambda *a, **k: None
        globals()["publish_mqtt"] = lambda *a, **k: None

# ---- 3) Image runner ----
btn_img = w.Button(description="Upload & Run (Images)", button_style="primary", icon="upload")

def _on_images(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No images uploaded."); return
            for name, data in up.items():
                try:
                    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
                    assert img is not None, f"decode failed: {name}"
                    dets, counts = run_yolo(img)

                    # density (masked or raw)
                    if mask_density.value:
                        d_local = density_sum_box_masked(img, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                    else:
                        _, d_local_raw = density_count_bgr(img)
                        d_local = float(d_local_raw)

                    y_local = float(counts.get("person",0))
                    animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                    use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)

                    # animal safeguard – prefer YOLO when animals are around and people exist
                    if animals_present and y_local > 0:
                        use_den = False

                    fused = (d_local if use_den else y_local)
                    src = "density" if use_den else "yolo"

                    vis = _draw_all(img, dets, counts, fused, src)

                    zones_people = {}
                    if ZONES:
                        zones_people, _ = zone_counts(img, dets, counts,
                                                      switch_to_density.value, dom_factor.value, min_valid.value,
                                                      use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                        vis = draw_zones(vis, zones_people)

                    # alerts
                    msgs=[]
                    if fused >= person_thr.value and should_alert("RETUNE_IMG"):
                        m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                        msgs.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"img_person_threshold","value":float(fused),"thr":person_thr.value,"ts":time.time()})
                    if animal_flag.value and fused>=1:
                        animals=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                        if animals and should_alert("RETUNE_IMG_ANIMAL"):
                            m=f"⚠️ Animal(s) in crowd: {', '.join(animals)} | persons≈{fused:.1f}"
                            msgs.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"img_animal_in_crowd","animals":animals,"people":float(fused),"ts":time.time()})

                    os.makedirs("/content/outputs/images", exist_ok=True)
                    out_path = f"/content/outputs/images/ANN_{os.path.basename(name)}"
                    cv2.imwrite(out_path, vis)

                    print(f"✅ {name} → {out_path}")
                    print(json.dumps({
                        "file": name,
                        "fused_persons": round(float(fused),1),
                        "fused_source": src,
                        "yolo_counts": {k:v for k,v in counts.items() if v>0},
                        "zones_persons": {k: round(v,1) for k,v in zones_people.items()} if zones_people else {},
                        **({"alerts": msgs} if msgs else {})
                    }, indent=2))
                    cv2_imshow(vis)
                except Exception as e:
                    print("[image error]", name, e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_img.on_click(_on_images)

# ---- 4) Video runner ----
btn_vid = w.Button(description="Upload & Run (Video)", button_style="primary", icon="upload")
show_every = w.IntSlider(value=6, min=1, max=30, step=1, description="Preview every N frames")
max_frames = w.IntText(value=0, description="Max frames (0=all)")
downscale_pixels = w.IntText(value=1280*720, description="Max pixels (downscale)")

def _on_video(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No video uploaded."); return
            vname = next(iter(up.keys()))
            vpath = f"/content/{vname}"; open(vpath,"wb").write(up[vname])

            cap = cv2.VideoCapture(vpath)
            assert cap.isOpened(), f"open failed: {vname}"
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

            scale = min(1.0, math.sqrt(downscale_pixels.value / max(1, W*H)))
            W2, H2 = max(64, int(W*scale)), max(64, int(H*scale))

            os.makedirs("/content/outputs/videos", exist_ok=True)
            out_path = f"/content/outputs/videos/processed_{int(time.time())}.mp4"
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W2,H2))

            n=0; alerts=[]
            t0=time.time()
            print(f"[run] {vname} → {out_path} | {W}x{H}@{fps:.1f} -> {W2}x{H2}")

            while True:
                ok, frame = cap.read()
                if not ok: break
                n+=1
                if max_frames.value and n>int(max_frames.value): break
                if scale < 0.999:
                    frame = cv2.resize(frame,(W2,H2), interpolation=cv2.INTER_AREA)

                dets, counts = run_yolo(frame)

                # density masked or raw
                if mask_density.value:
                    d_local = density_sum_box_masked(frame, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                else:
                    _, d_raw = density_count_bgr(frame); d_local=float(d_raw)

                y_local = float(counts.get("person",0))
                animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)
                if animals_present and y_local > 0:
                    use_den = False

                fused = (d_local if use_den else y_local)
                src = "density" if use_den else "yolo"

                vis = _draw_all(frame, dets, counts, fused, src)

                if 'ZONES' in globals() and ZONES:
                    zones_people, _ = zone_counts(frame, dets, counts,
                                                  switch_to_density.value, dom_factor.value, min_valid.value,
                                                  use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                    vis = draw_zones(vis, zones_people)
                    for z in ZONES:
                        thr = z.get("thr",{}).get("person", None)
                        if thr is not None and zones_people.get(z["name"],0) >= thr and should_alert(f"RETUNE_VID_ZONE_{z['name']}"):
                            m=f"🚨 Zone Alert [{z['name']}]: persons={zones_people[z['name']]:.1f} (≥{thr})"
                            alerts.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"video_zone_person_threshold","zone":z["name"],
                                                    "value":float(zones_people[z['name']]),"thr":thr,"ts":time.time()})

                # global alerts
                if fused >= person_thr.value and should_alert("RETUNE_VID"):
                    m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                    alerts.append(m); send_telegram(m, photo_bgr=vis)
                    publish_mqtt("alerts", {"type":"video_person_threshold","value":float(fused),
                                            "thr":person_thr.value,"ts":time.time()})
                if animal_flag.value and fused>=1:
                    a=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                    if a and should_alert("RETUNE_VID_ANIMAL"):
                        m=f"⚠️ Animal(s) in crowd: {', '.join(a)} | persons≈{fused:.1f}"
                        alerts.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"video_animal_in_crowd","animals":a,
                                                "people":float(fused),"ts":time.time()})

                if (n % max(1, int(show_every.value))) == 0:
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(vis)

                writer.write(vis)

            writer.release(); cap.release()
            dt=time.time()-t0
            print(f"\n[done] frames={n} | alerts={len(alerts)} | saved → {out_path} | {dt:.1f}s")
            print(json.dumps({"frames":n,"alerts":alerts,"output":out_path}, indent=2))
            try:
                display(HTML(f"""<video width="720" controls>
                  <source src="file://{out_path}" type="video/mp4"></video>"""))
            except: pass

        except Exception as e:
            print("[video error]", e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_vid.on_click(_on_video)

# ---- 5) Layout & display ----
row1 = w.HBox([person_thr, animal_flag])
row2 = w.HBox([switch_to_density, dom_factor, min_valid])
row3 = w.HBox([mask_density, expand_percent])
row4 = w.HBox([enable_external])
row5 = w.HBox([tg_token, tg_chat])
row6 = w.HBox([mqtt_host, mqtt_port, mqtt_topic])
row7 = w.HBox([w_path, btn_reload])
rowZ = w.HBox([btn_load_zones, zones_status])

tab_images = w.VBox([w.HTML("<b>Images</b>"), btn_img])
tab_video  = w.VBox([w.HTML("<b>Video</b>"), btn_vid, show_every, max_frames, downscale_pixels])

tabs = w.Tab(children=[tab_images, tab_video])
tabs.set_title(0, "Images"); tabs.set_title(1, "Video")

ui = w.VBox([hdr, row1, row2, row3, row4, row5, row6, rowZ, row7, tabs, out])
display(ui)


In [ ]:
# ==== RETUNE PATCH + CONFIGURABLE COLAB UI (uses your trained models) ====
# Assumes these are already defined from Steps 5–7:
# yolo, density_net, density_count_bgr, run_yolo, fused_people_count,
# send_telegram, publish_mqtt, should_alert, INTEREST_CLASSES, ANIMAL_CLASSES, (optional) ZONES

import os, cv2, json, math, time, traceback, numpy as np
from IPython.display import display, clear_output, HTML
from google.colab import files
import ipywidgets as w

# ---- 0) Reload your new CSRNet weights (safe if already loaded) ----
CSRNET_WEIGHTS = "/content/weights_csrnet_lite_stb_best.pt"
try:
    if 'density_net' in globals() and os.path.exists(CSRNET_WEIGHTS):
        density_net.load_state_dict(torch.load(CSRNET_WEIGHTS, map_location=next(density_net.parameters()).device))
        density_net.eval()
        print(f"[init] Reloaded CSRNet weights → {CSRNET_WEIGHTS}")
    else:
        print("[init] CSRNet weights not found or model missing; continuing with current state.")
except Exception as e:
    print("[init] reload failed:", e)

# ---- 1) Helpers (draw + density) ----
def _draw_all(vis, dets, counts_det, fused_global, fused_src):
    vis = vis.copy()
    for d in dets:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,f"{d['cls']} {d['conf']:.2f}",(x1,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)
    cv2.putText(vis,f"Persons (fused-{fused_src}): {fused_global:.1f}",(12,28),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2)
    return vis

def _polygon_mask(h, w, poly):
    m = np.zeros((h,w), dtype=np.uint8)
    pts = np.array(poly, dtype=np.int32)
    cv2.fillPoly(m, [pts], 1)
    return m.astype(bool)

def density_map_at_frame(frame_bgr):
    """Sum-preserving upsample of your density map to full frame."""
    dm_small, c_den = density_count_bgr(frame_bgr)
    hp, wp = dm_small.shape[:2]
    H, W = frame_bgr.shape[:2]
    dm_full = cv2.resize(dm_small.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    dm_full *= (hp * wp) / float(H * W)
    s = dm_full.sum()
    if s > 0:
        dm_full *= (c_den / s)
    return dm_full, float(c_den)

def density_sum_box_masked(img_bgr, dets, expand_ratio=0.15, require_person=True):
    """
    Sum density only inside YOLO 'person' boxes (expanded). If require_person and no boxes, return 0.
    """
    H, W = img_bgr.shape[:2]
    dm_full, _ = density_map_at_frame(img_bgr)
    mask = np.zeros((H,W), dtype=np.uint8)
    person_boxes = [d["bbox"] for d in dets if d["cls"] == "person"]
    if require_person and not person_boxes:
        return 0.0
    for (x1,y1,x2,y2) in person_boxes:
        dx = int((x2-x1) * expand_ratio); dy = int((y2-y1) * expand_ratio)
        x1e = max(0, x1-dx); y1e = max(0, y1-dy)
        x2e = min(W-1, x2+dx); y2e = min(H-1, y2+dy)
        mask[y1e:y2e+1, x1e:x2e+1] = 1
    return float(dm_full[mask.astype(bool)].sum())

def zone_counts(frame_bgr, dets, counts_det,
                switch_to_density=10.0, dom_factor=1.2, min_valid=5.0,
                use_mask=True, expand_ratio=0.15):
    """Zone-wise fused counts with optional density masking."""
    if 'ZONES' not in globals() or not ZONES:
        return {}, {}
    H, W = frame_bgr.shape[:2]

    # YOLO per zone
    zone_classes = {z["name"]: {k:0 for k in INTEREST_CLASSES} for z in ZONES}
    for d in dets:
        cx, cy = d["center"]
        if not (0 <= cx < W and 0 <= cy < H): continue
        for z in ZONES:
            pts = np.array(z["poly"], np.int32)
            if cv2.pointPolygonTest(pts, (float(cx), float(cy)), False) >= 0:
                cname = d["cls"]
                if cname in zone_classes[z["name"]]:
                    zone_classes[z["name"]][cname] += 1

    # Density per zone
    dm_full, _ = density_map_at_frame(frame_bgr)
    zone_people_density = {}
    for z in ZONES:
        mask = _polygon_mask(H, W, z["poly"])
        if use_mask:
            # combine with person boxes
            m2 = np.zeros((H,W), dtype=np.uint8)
            for d in dets:
                if d["cls"] != "person": continue
                x1,y1,x2,y2 = d["bbox"]
                m2[y1:y2+1, x1:x2+1] = 1
            zmask = (mask & m2.astype(bool))
        else:
            zmask = mask
        zone_people_density[z["name"]] = float(dm_full[zmask].sum())

    # Fused per zone
    zone_people_fused={}
    for z in ZONES:
        y = float(zone_classes[z["name"]].get("person", 0))
        d = float(zone_people_density[z["name"]])
        use_den = (d >= max(min_valid, switch_to_density)) or (d > y * dom_factor)
        zone_people_fused[z["name"]] = (d if use_den else y)
    return zone_people_fused, zone_classes

def draw_zones(vis, zones_people):
    if 'ZONES' not in globals() or not ZONES: return vis
    vis = vis.copy()
    for i, z in enumerate(ZONES):
        color = (0,255,255) if i%2==0 else (255,128,0)
        pts = np.array(z["poly"], dtype=np.int32)
        cv2.polylines(vis, [pts], True, color, 2)
        cx = int(np.mean([p[0] for p in z["poly"]])); cy = int(np.mean([p[1] for p in z["poly"]]))
        cv2.putText(vis, f"{z['name']}: {zones_people.get(z['name'],0):.1f}", (cx-60, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return vis

# ---- 2) ipywidgets UI with retuned defaults (density is stronger now) ----
hdr = w.HTML("<h3>Crowd Count · YOLOv12 + Density Fusion · Retuned Interface</h3>")

# Fusion/alert defaults tuned for your 20-epoch model
person_thr = w.IntSlider(value=50, min=1, max=500, step=1, description="Global threshold")
animal_flag = w.Checkbox(value=True, description="Alert if animal in crowd")

switch_to_density = w.FloatSlider(value=8.0,  min=0, max=150, step=0.5, description="Switch@≥ density")
dom_factor       = w.FloatSlider(value=1.15, min=1.0, max=3.0, step=0.05, description="Density > YOLO ×")
min_valid        = w.FloatSlider(value=4.0,  min=0, max=50,  step=0.5, description="Min valid density")

mask_density   = w.Checkbox(value=True, description="Mask density to YOLO person boxes")
expand_percent = w.FloatSlider(value=15.0, min=0, max=50, step=1.0, description="Box expand %")

# External alerts (optional)
enable_external = w.Checkbox(value=True, description="Enable external alerts (Telegram/MQTT)")
tg_token = w.Text(value=globals().get("BOT_TOKEN",""), description="Telegram BOT_TOKEN", layout=w.Layout(width="60%"))
tg_chat  = w.Text(value=str(globals().get("CHAT_ID","")), description="Telegram CHAT_ID", layout=w.Layout(width="40%"))
mqtt_host = w.Text(value=globals().get("MQTT_HOST","broker.emqx.io"), description="MQTT host", layout=w.Layout(width="45%"))
mqtt_port = w.IntText(value=globals().get("MQTT_PORT",1883), description="Port", layout=w.Layout(width="20%"))
mqtt_topic= w.Text(value=globals().get("MQTT_TOPIC_BASE","site/demo/camera/colab"), description="MQTT topic base", layout=w.Layout(width="35%"))

# Zones
ZONES = globals().get("ZONES", [])
zones_status = w.HTML(value=f"<b>Zones:</b> {'Loaded' if ZONES else 'None'}")
def _load_zones(_):
    global ZONES
    if os.path.exists("zones.json"):
        try:
            with open("zones.json","r") as f: ZONES = json.load(f)
            zones_status.value = f"<b>Zones:</b> Loaded {len(ZONES)}"
        except Exception as e:
            zones_status.value = f"<b>Zones:</b> read error ({e})"
    else:
        zones_status.value = "<b>Zones:</b> zones.json not found"
btn_load_zones = w.Button(description="Load zones.json"); btn_load_zones.on_click(_load_zones)

# Reload weights button
w_path = w.Text(value=CSRNET_WEIGHTS, description="CSRNet weights", layout=w.Layout(width="70%"))
def _reload(_):
    try:
        density_net.load_state_dict(torch.load(w_path.value, map_location=next(density_net.parameters()).device))
        density_net.eval(); print(f"[ok] reloaded {w_path.value}")
    except Exception as e:
        print("[reload err]", e)
btn_reload = w.Button(description="Reload weights", button_style="")

# Output
out = w.Output()

# Utils to wire external creds
def _apply_external_settings():
    global BOT_TOKEN, CHAT_ID, MQTT_HOST, MQTT_PORT, MQTT_TOPIC_BASE
    BOT_TOKEN = tg_token.value.strip()
    try: CHAT_ID = int(tg_chat.value.strip()) if tg_chat.value.strip() else 0
    except: CHAT_ID = 0
    MQTT_HOST = mqtt_host.value.strip()
    MQTT_PORT = int(mqtt_port.value)
    MQTT_TOPIC_BASE = mqtt_topic.value.strip()

def _toggle_alert_shims(enable: bool):
    """No-op the senders when disabled."""
    global _send_real, _publish_real
    if enable:
        if "_send_real" in globals(): globals()["send_telegram"] = _send_real
        if "_publish_real" in globals(): globals()["publish_mqtt"] = _publish_real
    else:
        if "send_telegram" in globals() and "_send_real" not in globals():
            _send_real = globals()["send_telegram"]
        if "publish_mqtt" in globals() and "_publish_real" not in globals():
            _publish_real = globals()["publish_mqtt"]
        globals()["send_telegram"] = lambda *a, **k: None
        globals()["publish_mqtt"] = lambda *a, **k: None

# ---- 3) Image runner ----
btn_img = w.Button(description="Upload & Run (Images)", button_style="primary", icon="upload")

def _on_images(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No images uploaded."); return
            for name, data in up.items():
                try:
                    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
                    assert img is not None, f"decode failed: {name}"
                    dets, counts = run_yolo(img)

                    # density (masked or raw)
                    if mask_density.value:
                        d_local = density_sum_box_masked(img, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                    else:
                        _, d_local_raw = density_count_bgr(img)
                        d_local = float(d_local_raw)

                    y_local = float(counts.get("person",0))
                    animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                    use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)

                    # animal safeguard – prefer YOLO when animals are around and people exist
                    if animals_present and y_local > 0:
                        use_den = False

                    fused = (d_local if use_den else y_local)
                    src = "density" if use_den else "yolo"

                    vis = _draw_all(img, dets, counts, fused, src)

                    zones_people = {}
                    if ZONES:
                        zones_people, _ = zone_counts(img, dets, counts,
                                                      switch_to_density.value, dom_factor.value, min_valid.value,
                                                      use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                        vis = draw_zones(vis, zones_people)

                    # alerts
                    msgs=[]
                    if fused >= person_thr.value and should_alert("RETUNE_IMG"):
                        m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                        msgs.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"img_person_threshold","value":float(fused),"thr":person_thr.value,"ts":time.time()})
                    if animal_flag.value and fused>=1:
                        animals=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                        if animals and should_alert("RETUNE_IMG_ANIMAL"):
                            m=f"⚠️ Animal(s) in crowd: {', '.join(animals)} | persons≈{fused:.1f}"
                            msgs.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"img_animal_in_crowd","animals":animals,"people":float(fused),"ts":time.time()})

                    os.makedirs("/content/outputs/images", exist_ok=True)
                    out_path = f"/content/outputs/images/ANN_{os.path.basename(name)}"
                    cv2.imwrite(out_path, vis)

                    print(f"✅ {name} → {out_path}")
                    print(json.dumps({
                        "file": name,
                        "fused_persons": round(float(fused),1),
                        "fused_source": src,
                        "yolo_counts": {k:v for k,v in counts.items() if v>0},
                        "zones_persons": {k: round(v,1) for k,v in zones_people.items()} if zones_people else {},
                        **({"alerts": msgs} if msgs else {})
                    }, indent=2))
                    cv2_imshow(vis)
                except Exception as e:
                    print("[image error]", name, e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_img.on_click(_on_images)

# ---- 4) Video runner ----
btn_vid = w.Button(description="Upload & Run (Video)", button_style="primary", icon="upload")
show_every = w.IntSlider(value=6, min=1, max=30, step=1, description="Preview every N frames")
max_frames = w.IntText(value=0, description="Max frames (0=all)")
downscale_pixels = w.IntText(value=1280*720, description="Max pixels (downscale)")

def _on_video(_):
    from google.colab.patches import cv2_imshow
    with out:
        clear_output(wait=True)
        try:
            _apply_external_settings(); _toggle_alert_shims(enable_external.value)
            up = files.upload()
            if not up: print("No video uploaded."); return
            vname = next(iter(up.keys()))
            vpath = f"/content/{vname}"; open(vpath,"wb").write(up[vname])

            cap = cv2.VideoCapture(vpath)
            assert cap.isOpened(), f"open failed: {vname}"
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

            scale = min(1.0, math.sqrt(downscale_pixels.value / max(1, W*H)))
            W2, H2 = max(64, int(W*scale)), max(64, int(H*scale))

            os.makedirs("/content/outputs/videos", exist_ok=True)
            out_path = f"/content/outputs/videos/processed_{int(time.time())}.mp4"
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W2,H2))

            n=0; alerts=[]
            t0=time.time()
            print(f"[run] {vname} → {out_path} | {W}x{H}@{fps:.1f} -> {W2}x{H2}")

            while True:
                ok, frame = cap.read()
                if not ok: break
                n+=1
                if max_frames.value and n>int(max_frames.value): break
                if scale < 0.999:
                    frame = cv2.resize(frame,(W2,H2), interpolation=cv2.INTER_AREA)

                dets, counts = run_yolo(frame)

                # density masked or raw
                if mask_density.value:
                    d_local = density_sum_box_masked(frame, dets, expand_ratio=expand_percent.value/100.0, require_person=False)
                else:
                    _, d_raw = density_count_bgr(frame); d_local=float(d_raw)

                y_local = float(counts.get("person",0))
                animals_present = any(k in ANIMAL_CLASSES and v>0 for k,v in counts.items())
                use_den = (d_local >= max(min_valid.value, switch_to_density.value)) or (d_local > y_local * dom_factor.value)
                if animals_present and y_local > 0:
                    use_den = False

                fused = (d_local if use_den else y_local)
                src = "density" if use_den else "yolo"

                vis = _draw_all(frame, dets, counts, fused, src)

                if 'ZONES' in globals() and ZONES:
                    zones_people, _ = zone_counts(frame, dets, counts,
                                                  switch_to_density.value, dom_factor.value, min_valid.value,
                                                  use_mask=mask_density.value, expand_ratio=expand_percent.value/100.0)
                    vis = draw_zones(vis, zones_people)
                    for z in ZONES:
                        thr = z.get("thr",{}).get("person", None)
                        if thr is not None and zones_people.get(z["name"],0) >= thr and should_alert(f"RETUNE_VID_ZONE_{z['name']}"):
                            m=f"🚨 Zone Alert [{z['name']}]: persons={zones_people[z['name']]:.1f} (≥{thr})"
                            alerts.append(m); send_telegram(m, photo_bgr=vis)
                            publish_mqtt("alerts", {"type":"video_zone_person_threshold","zone":z["name"],
                                                    "value":float(zones_people[z['name']]),"thr":thr,"ts":time.time()})

                # global alerts
                if fused >= person_thr.value and should_alert("RETUNE_VID"):
                    m=f"🚨 Crowd Alert: persons={fused:.1f} (≥{person_thr.value})"
                    alerts.append(m); send_telegram(m, photo_bgr=vis)
                    publish_mqtt("alerts", {"type":"video_person_threshold","value":float(fused),
                                            "thr":person_thr.value,"ts":time.time()})
                if animal_flag.value and fused>=1:
                    a=[k for k,v in counts.items() if k in ANIMAL_CLASSES and v>0]
                    if a and should_alert("RETUNE_VID_ANIMAL"):
                        m=f"⚠️ Animal(s) in crowd: {', '.join(a)} | persons≈{fused:.1f}"
                        alerts.append(m); send_telegram(m, photo_bgr=vis)
                        publish_mqtt("alerts", {"type":"video_animal_in_crowd","animals":a,
                                                "people":float(fused),"ts":time.time()})

                if (n % max(1, int(show_every.value))) == 0:
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(vis)

                writer.write(vis)

            writer.release(); cap.release()
            dt=time.time()-t0
            print(f"\n[done] frames={n} | alerts={len(alerts)} | saved → {out_path} | {dt:.1f}s")
            print(json.dumps({"frames":n,"alerts":alerts,"output":out_path}, indent=2))
            try:
                display(HTML(f"""<video width="720" controls>
                  <source src="file://{out_path}" type="video/mp4"></video>"""))
            except: pass

        except Exception as e:
            print("[video error]", e); traceback.print_exc()
        finally:
            if enable_external.value: _toggle_alert_shims(True)

btn_vid.on_click(_on_video)

# ---- 5) Layout & display ----
row1 = w.HBox([person_thr, animal_flag])
row2 = w.HBox([switch_to_density, dom_factor, min_valid])
row3 = w.HBox([mask_density, expand_percent])
row4 = w.HBox([enable_external])
row5 = w.HBox([tg_token, tg_chat])
row6 = w.HBox([mqtt_host, mqtt_port, mqtt_topic])
row7 = w.HBox([w_path, btn_reload])
rowZ = w.HBox([btn_load_zones, zones_status])

tab_images = w.VBox([w.HTML("<b>Images</b>"), btn_img])
tab_video  = w.VBox([w.HTML("<b>Video</b>"), btn_vid, show_every, max_frames, downscale_pixels])

tabs = w.Tab(children=[tab_images, tab_video])
tabs.set_title(0, "Images"); tabs.set_title(1, "Video")

ui = w.VBox([hdr, row1, row2, row3, row4, row5, row6, rowZ, row7, tabs, out])
display(ui)
